# Qwen3-ASR Transcription

A lightweight pipeline to transcribe audio files to text using [Qwen3-ASR](https://huggingface.co/Qwen/Qwen3-ASR-0.6B).

**Steps:**
1. Open Google Colab (T4 GPU)
2. Install dependencies
3. Load the ASR model
4. Upload an audio file
5. Transcribe
6. Save the output

## Block 1 — Open Google Colab with T4 GPU

Before running any code, make sure you're using a GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** under "Hardware accelerator"
3. Click **Save**

Why GPU? The ASR model runs much faster on GPU. CPU works but is significantly slower.

In [ ]:
import torch

# Check GPU — warn if not available
if not torch.cuda.is_available():
    print("WARNING: No GPU detected.")
    print("Go to Runtime → Change runtime type → T4 GPU")
    print("The model will run on CPU but will be significantly slower.")
else:
    print("GPU detected:", torch.cuda.get_device_name(0))

## Block 2 — Install dependencies

Install the ASR package and audio processing libraries. The `-q` flag keeps output minimal.

`ffmpeg` is needed for non-WAV formats like .mp3, .m4a, and .ogg — without it, these files will fail to load.

In [ ]:
# Install ffmpeg for .mp3, .m4a, .ogg support
!apt-get install -y -q ffmpeg

# Install qwen-asr (includes torch/torchaudio) and audio processing libs
!pip install -q qwen-asr librosa soundfile

## Block 3 — Import modules

Import the required libraries for the ASR pipeline.

In [ ]:
import librosa
import soundfile as sf

from qwen_asr import Qwen3ASRModel

# Reuse device from Block 1's GPU check
device = "cuda" if torch.cuda.is_available() else "cpu"

## Block 4 — Load the Qwen3-ASR model

Choose `Qwen/Qwen3-ASR-0.6B` (faster, lower VRAM) or `Qwen/Qwen3-ASR-1.7B` (higher accuracy, more VRAM).

- `bfloat16` is used on GPU to save memory (falls back to `float16` on older GPUs)
- `float32` is used on CPU since bfloat16/float16 aren't supported there
- `device_map="auto"` lets the library place the model on the GPU automatically

In [ ]:
# Switch to "Qwen/Qwen3-ASR-1.7B" for higher accuracy (needs more VRAM)
model_name = "Qwen/Qwen3-ASR-0.6B"

# Check VRAM before loading — warn if 1.7B model might OOM
if device == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Available VRAM: {vram_gb:.1f} GB")
    if "1.7B" in model_name and vram_gb < 16:
        print("WARNING: 1.7B model may OOM on this GPU. Consider switching to 0.6B.")

# Choose best dtype for GPU — bfloat16 preferred, float16 fallback, float32 for CPU
if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

asr_model = Qwen3ASRModel.from_pretrained(
    model_name,
    dtype=dtype,
    device_map="auto" if device == "cuda" else None
)

print("ASR model loaded:", model_name)

## Block 5 — Upload and play the audio file

Upload any audio file (.wav, .mp3, .m4a, etc.). The file will play back in Colab so you can verify it loaded correctly.

**Sample files included:** If you don't have an audio file ready, use the samples below (English or Chinese).

In [ ]:
from google.colab import files
from IPython.display import Audio, display
import os

# Option 1: Upload your own audio file
# uploaded = files.upload()
# audio_file = list(uploaded.keys())[0]

# Option 2: Use a sample file from the repo (uncomment one)
# !wget -q https://raw.githubusercontent.com/wsamuelw/qwen3-asr/main/sample-voice/english-sample-voice.wav -O sample.wav
# !wget -q https://raw.githubusercontent.com/wsamuelw/qwen3-asr/main/sample-voice/chinese-sample-voice.wav -O sample.wav
# audio_file = "sample.wav"

print("Audio file:", audio_file)

# Play the audio so you can verify it loaded correctly
display(Audio(audio_file))

## Block 6 — Convert to mono 16kHz

The Qwen3-ASR model requires audio in 16kHz mono format. This step resamples and converts any input audio to match.

- `librosa.load()` handles format conversion and resampling automatically
- The cleaned audio is saved as `clean_audio.wav` for the next step

In [ ]:
# Resample to 16kHz mono — the model's required input format
try:
    audio, sr = librosa.load(audio_file, sr=16000, mono=True)
except Exception as e:
    print(f"Failed to load audio: {e}")
    print("Try a different file format (.wav, .mp3, .m4a)")
    raise

clean_path = "clean_audio.wav"
sf.write(clean_path, audio, 16000)

print("Normalized audio saved as:", clean_path)

## Block 7 — Transcribe the audio

Run the ASR model on the cleaned audio. Key options:

- `language=None` — automatic language detection (supports 20+ languages)
- `return_time_stamps=False` — set to `True` if you want word-level timing
- Results come back as a list; `results[0].text` gives the transcription string

In [ ]:
import time

# Run transcription with inference_mode to save VRAM and speed up
start = time.time()
try:
    with torch.inference_mode():
        results = asr_model.transcribe(
            audio=clean_path,
            language=None,
            return_time_stamps=False
        )
except Exception as e:
    print(f"Transcription failed: {e}")
    print("Try a shorter audio file or switch to the 0.6B model")
    raise

elapsed = time.time() - start

# Extract the text from the first (and only) result
transcript = results[0].text

print("Transcription output:")
print(transcript)
print(f"\nTranscribed {len(audio)/sr:.1f}s of audio in {elapsed:.1f}s")

## Block 8 — Save transcription

Save the transcription to a text file. This is useful for:
- Archiving transcriptions for later analysis
- Using as training data for other NLP tasks
- Keeping a record alongside the original audio

In [ ]:
# Save transcription to file
with open("transcription.txt", "w") as f:
    f.write(transcript)

print("Transcription saved to transcription.txt")